# AQCat AWS Marketplace Sample Notebook

Clone-and-run sample for the **AQCat** SageMaker model package. It takes you from a
Marketplace subscription to live predictions and back to a clean account.

**Outline:** 

1) subscribe
2) set up
3) create model
4) real-time (adsorption + MLP) 
5) clean up

> 💰 The notebook creates **billable** resources. **Run section 5 (Clean up)** when you finish to avoid leaving a forgotten endpoint running and billing continuously. Billing is metered per successful response via `X-Amzn-Inference-Metering`, with every call of the MLP resulting in one inference metered (whether explicitly called in `mlp` mode or called multiple times in `min-adsorption-energy-workflow` mode).

> ⚠️ If you have launched the product through Sagemaker Studio, you may have already created an endpoint. We suggest you delete it right away, as this notebook will create and delete an AQCat endpoint. You can manage existing endpoints in SageMaker Studio by looking at the side menu on the left (Deployment > Endpoints).

## 1. Subscribe

Subscribe to the [AQCat listing](https://aws.amazon.com/marketplace/pp/prodview-smq2zrd5oqpnc) in the AWS web console. Once you're at the listing page, click `View purchase options` > then `Subscribe` on the next page.

Then paste in the ARN for your region. You can find this by going to `AWS Marketplace` > `Manage subscriptions`, find the `AQCat` subscription and click the `Configure` action (see image below). Select `Amazon SageMaker AI console` > Scroll to the bottom to find `Model ARNs` listed (see `region_arn_1.png` and `region_arn_2.png` for more details, alongside this notebook).

In [ ]:
model_package_arn = '' # paste your region ARN here

## 2. Set up your environment

### Option A — Amazon SageMaker Studio web UI (fastest)

- Launch [SageMaker Studio](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-launch.html).
- Clone this repo (or upload [the notebook](./aqcat-sample.ipynb)) into Studio.
- Open the notebook and run the cells top to bottom.

### Option B — local / other environment
- Clone this repo into your environment and navigate to this directory.
- Ensure you have the packages specified in [requirements.txt](./requirements.txt) installed in your environment. Otherwise, you can run:
    ```bash
    pip install -r requirements.txt
    aws configure          # credentials + default region
    ```

Then run the code below to set up the session.

> ⚠️ Make sure the region printed by the cell below indeed matches the one you used for the model arn.

In [ ]:
import json
import time
import boto3

# Try the imports for SageMaker SDK v3 first, if not available, fallback to v2.
try:
    from sagemaker.core.helper.session_helper import Session, get_execution_role  # SDK v3
except ImportError:
    from sagemaker import Session, get_execution_role                             # SDK v2

session = Session()
region  = session.boto_region_name
role    = get_execution_role()

sagemaker_client = boto3.client('sagemaker', region_name=region)          # create/delete model + endpoint
runtime          = boto3.client('sagemaker-runtime', region_name=region)  # invoke the endpoint
s3               = boto3.client('s3', region_name=region)

print('Region :', region)
print('Role   :', role)
print('Using model package:', model_package_arn)

## 3. Create the model in SageMaker

This creates a SageMaker **model** from the Marketplace package you subscribed to, referenced by its arn. We call the `boto3` SageMaker client directly, network isolation is required for Marketplace model packages.

In [ ]:
model_name = 'aqcat-sample-model'

sagemaker_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    Containers=[{'ModelPackageName': model_package_arn}],
    EnableNetworkIsolation=True,   # required for AWS Marketplace model packages
)
print('Created model:', model_name)

## 4. Real-time inference (recommended)

In order to query the model, we must deploy an endpoint. For AQCat, a single endpoint serves both modes:
`mode:"min-adsorption-energy-workflow"` in §4a and `mode:"mlp"` in §4b. Deployment takes a few
minutes. (~ 5 mins)

In [ ]:
endpoint_name        = 'aqcat-sample-endpoint'
endpoint_config_name = 'aqcat-sample-config'

sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': model_name,
        'InstanceType': 'ml.g5.xlarge',   # see recommendations in hardware_selection_guide.md
        'InitialInstanceCount': 1,
    }],
)
sagemaker_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print('Deploying endpoint — this takes a few minutes…')
sagemaker_client.get_waiter('endpoint_in_service').wait(EndpointName=endpoint_name)
print('Endpoint live:', endpoint_name)

### 4a. MLP single inference (`mode:"mlp"`)

Send a raw geometry as an ASE `Atoms` object — chemical **symbols** (`"H"`) + **positions** in Å,
plus the `cell`/`pbc` an `Atoms` carries (all-zeros and non-periodic for an isolated molecule) — and
get back the total **energy** (eV) and per-atom **forces** (eV/Å) from a single forward pass. The
`instances` field can include multiple objects to run a mini-batch (limit mini-batch sizes according
to memory constraints detailed in the [Hardware Selection Guide](./hardware_selection_guide.md)).

In [ ]:
mlp_payload = {
    'mode': 'mlp',                       # the default mode; shown explicitly for clarity
    'instances': [
        {                                # one ASE Atoms object; add more for a mini-batch
            'symbols': ['H', 'H'],       # chemical symbols, one per atom
            'positions': [[0.0, 0.0, 0.0], [0.74, 0.0, 0.0]],  # x, y, z in Angstrom
            'cell': [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],  # all-zeros for a molecule
            'pbc': [False, False, False],                                 # non-periodic
        }
    ],
    'parameters': {},                    # MLP mode has no tunable parameters
}

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Accept='application/json',
    Body=json.dumps(mlp_payload),
)
result = json.loads(response['Body'].read())
print(json.dumps(result, indent=2))

pred = result['predictions'][0]
print(f"\nenergy = {pred['energy']} eV")
print(f"forces = {pred['forces']} eV/Angstrom")

### 4b. Adsorption workflow (`mode:"min-adsorption-energy-workflow"`)

The adsorption workflow calculates min adsorption energy. You give it a bulk material (by ID or composition), a surface facet, and an adsorbate; it builds the surface, tries several adsorbate placements, relaxes each one, and returns the relaxed **energy** plus quality flags. 

The request is `instances` (what to screen) + `parameters` (how to relax). The response has one
prediction per placement; the lowest `energy` is the predicted binding configuration. Because `mlp` is the default mode, this mode **must** be set explicitly. 

In [ ]:
workflow_payload = {
    'mode': 'min-adsorption-energy-workflow',  # required — mlp is the default mode
    'instances': [
        {
            'bulk_src_ids': ['mp-102'],  # bulk material (Materials Project ID)
            'facets': [[1, 1, 1]],        # surface facet (Miller indices)
            'adsorbates': ['*CO'],        # adsorbate (* = adsorbed species)
            'num_placements': 3,          # placements to try (1-5)
        }
    ],
    'parameters': {'seed': 42, 'fmax': 0.05, 'max_steps': 150, 'converged_placements_only': True},
}

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Accept='application/json',
    Body=json.dumps(workflow_payload),
)
result = json.loads(response['Body'].read())
print(json.dumps(result, indent=2))

preds = result['predictions']
best = min(preds, key=lambda p: p['energy'])
print(f"\n{len(preds)} placements; best energy = {best['energy']} eV "
      f"(placement {best['placement_idx']}, relaxed={best['is_relaxed']})")

## (Not recommended) Batch transform

The real time inference endpoint provides the capability to run batches. Batch transform is provided as required by AWS, but not supported or recommended.

Batch transform is for scoring many inputs that are known beforehand and stored in an S3 bucket. As a result, it is not a good fit for optimization workflows, where the inputs for a given forward pass depend on the outputs of the previous one. 

An example of how to use batch transform in mode `min-adsorption-energy-workflow` (the adsorption workflow) is below. Each input file is **one complete request body** (and may hold many `instances`); the job writes one `*.out` file per input file. A small helper runs a batch job from an inline request body and prints the results.

This cell may take up to 10 mins to complete.

In [ ]:
def run_batch(request_body, key_prefix):
    """Upload one request file to S3, run a batch transform job, and print every result file."""
    account = boto3.client('sts', region_name=region).get_caller_identity()['Account']
    bucket  = f'sagemaker-{region}-{account}'   # SageMaker's default bucket naming convention
    try:
        s3.head_bucket(Bucket=bucket)
    except s3.exceptions.ClientError:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket)
        else:
            s3.create_bucket(Bucket=bucket, CreateBucketConfiguration={'LocationConstraint': region})

    input_key   = f'{key_prefix}/request_001.json'
    output_pref = f'{key_prefix}-output'
    s3.put_object(Bucket=bucket, Key=input_key, Body=json.dumps(request_body).encode())
    input_s3  = f's3://{bucket}/{input_key}'
    output_s3 = f's3://{bucket}/{output_pref}'
    print('Uploaded:', input_s3)

    job_name = f'aqcat-batch-{int(time.time())}'
    sagemaker_client.create_transform_job(
        TransformJobName=job_name,
        ModelName=model_name,
        BatchStrategy='SingleRecord',
        MaxConcurrentTransforms=1,
        MaxPayloadInMB=6,
        TransformInput={
            'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': input_s3}},
            'ContentType': 'application/json',
            'SplitType': 'None',
        },
        TransformOutput={'S3OutputPath': output_s3, 'Accept': 'application/json'},
        TransformResources={'InstanceType': 'ml.g4dn.xlarge', 'InstanceCount': 1},
    )
    sagemaker_client.get_waiter('transform_job_completed_or_stopped').wait(TransformJobName=job_name)

    for obj in s3.list_objects_v2(Bucket=bucket, Prefix=output_pref).get('Contents', []):
        if obj['Key'].endswith('.out'):
            body = s3.get_object(Bucket=bucket, Key=obj['Key'])['Body'].read().decode()
            print(obj['Key'].split('/')[-1], '->', body)

# One request body per file. This one screens four bulk/facet/adsorbate combinations.
workflow_batch_request = {
    'mode': 'min-adsorption-energy-workflow',  # required — mlp is the default mode
    'instances': [
        {'bulk_src_ids': ['mp-102'], 'facets': [[1, 1, 1]], 'adsorbates': ['*CO'], 'num_placements': 1},
        {'bulk_src_ids': ['mp-102'], 'facets': [[1, 0, 0]], 'adsorbates': ['*CO'], 'num_placements': 1},
        {'bulk_src_ids': ['mp-102'], 'facets': [[1, 1, 1]], 'adsorbates': ['*OH'], 'num_placements': 1},
        {'bulk_composition_contains': 'Cu', 'facets': [[1, 1, 1]], 'adsorbates': ['*CO'], 'num_placements': 1},
    ],
    'parameters': {'seed': 42, 'fmax': 0.05, 'max_steps': 150, 'converged_placements_only': True},
}

run_batch(workflow_batch_request, key_prefix='aqcat/workflow-batch-input')

## 5. Clean up

Delete the endpoint and model to stop infrastructure charges. **Run this when you finish.** To stop
the software charge entirely, also **unsubscribe** under AWS Marketplace → Manage subscriptions.

_Billing note: each successful response is metered via `X-Amzn-Inference-Metering`.
Adsorption-workflow mode (`min-adsorption-energy-workflow`): `ConsumedUnits` = the number of model
forward passes (MLP calls) — one per optimizer step across all
`bulks × facets × adsorbates × num_placements` relaxations (≤ `… × max_steps`). MLP mode:
`ConsumedUnits` = number of instances. Failed requests are not billed. See the
[Billing guide](./billing_guide.md)._

In [ ]:
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sagemaker_client.delete_model(ModelName=model_name)
print('Cleaned up endpoint, endpoint config, and model.')